# Customizing the OTC Cycle

The observe-think-act (OTC) cycle is the heartbeat of every amphibious agent. At each stage, the framework provides hook methods you can override to inject custom behavior — whether it's enriching observations, reshaping LLM messages, intercepting tool calls, or post-processing outputs.

These hooks exist at two levels: **Worker level** (per-worker customization) and **Agent level** (shared across all workers). When a worker hook returns the special `_DELEGATE` sentinel, the framework falls through to the agent-level hook.

In this tutorial, we'll build a **security audit agent** that uses OTC hooks to inject system state, enforce safety rules, filter dangerous operations, and sanitize output.

## Initialize

First, let's set up the LLM and define the security-themed tools our audit agent will use.

In [ ]:
import os
from bridgic.model import OpenAILlm

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = LLM(model="openai/gpt-4o-mini")

In [ ]:
from bridgic.core import tool


@tool(description="List files in a directory")
async def list_files(directory: str) -> str:
    return f"Files in {directory}: config.yaml, app.log, secrets.env, data.db"


@tool(description="Read the content of a file")
async def read_file(filepath: str) -> str:
    if "secret" in filepath.lower() or ".env" in filepath:
        return f"[SENSITIVE] Content of {filepath}: API_KEY=sk-xxx, DB_PASS=yyy"
    return f"Content of {filepath}: (normal file content)"


@tool(description="Delete a file from the system")
async def delete_file(filepath: str) -> str:
    return f"Deleted {filepath}"


@tool(description="Check system security status")
async def check_security_status() -> str:
    return "System status: 2 warnings, 0 critical, last scan: 1 hour ago"

We have four tools:

- **`list_files`** — lists files in a given directory. Returns a mix of normal and sensitive files.
- **`read_file`** — reads a file's content. If the file is sensitive (e.g., `.env` or `secrets`), it returns data containing API keys and passwords.
- **`delete_file`** — deletes a file. This is the dangerous operation we want to block.
- **`check_security_status`** — reports the system's current security posture.

## Part 1: Observe Phase — Injecting Custom Perception

The `observation()` hook lets you inject custom context before the LLM starts thinking. It exists at two levels:

- **Worker-level `observation()`**: Gives a specific worker its own view of the environment — for example, injecting security policies or audit metadata.
- **Agent-level `observation()`**: Provides shared context for all workers — for example, system-wide information like server name and region.
- When a worker returns `_DELEGATE`, the framework falls through to the agent's `observation()` instead.

Let's build a security worker that injects audit-specific context into the observe phase.

In [ ]:
from bridgic.amphibious import (
    AmphibiousAutoma, CognitiveContext, CognitiveWorker,
    think_unit, _DELEGATE,
)


class SecurityWorker(CognitiveWorker):
    """A worker with custom observation that injects security logs."""

    async def thinking(self) -> str:
        return (
            "Analyze the system for security issues. "
            "Check files, review the security status, and report findings. "
            "NEVER delete files or access sensitive files like .env or secrets."
        )

    async def observation(self, context: CognitiveContext):
        """Inject security-specific context before thinking."""
        return (
            f"Current goal: {context.goal}\n"
            f"Security policy: Read-only audit mode. File deletion is PROHIBITED.\n"
            f"Audit timestamp: 2024-06-15T10:30:00Z"
        )


class SecurityAuditAgent(AmphibiousAutoma[CognitiveContext]):
    auditor = think_unit(SecurityWorker(), max_attempts=5)

    async def observation(self, ctx: CognitiveContext):
        """Agent-level observation: provides system-wide context for all workers."""
        return f"System: production-server-01, Region: us-east-1, Uptime: 45 days"

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor

In [ ]:
agent = SecurityAuditAgent(llm=llm)
result = await agent.arun(
    goal="Perform a security audit on the /var/app directory",
    tools=[list_files, read_file, delete_file, check_security_status],
)
print(result)

Notice how the `SecurityWorker` injects its own observation — the security policy and audit timestamp — into the context before the LLM reasons about the task. Because the worker defines its own `observation()`, it does **not** fall through to the agent-level observation.

If you wanted a worker to use the agent-level observation instead, you would return `_DELEGATE` from the worker's `observation()` method:

```python
async def observation(self, context: CognitiveContext):
    return _DELEGATE  # Fall through to agent-level observation
```

This two-tier pattern lets you share common context across workers while allowing individual workers to override it when needed.

## Part 2: Think Phase — Reshaping LLM Messages

The `build_messages()` hook controls how the prompt is assembled before being sent to the LLM. By overriding it, you can inject mandatory rules, restructure the message format, or add custom system instructions.

Let's create a worker that injects strict security rules into every LLM call.

In [ ]:
from bridgic.amphibious._cognitive_worker import Message


class StrictSecurityWorker(CognitiveWorker):
    async def thinking(self) -> str:
        return "Perform a security audit of the specified directory."

    async def build_messages(
        self,
        think_prompt: str,
        tools_description: str,
        output_instructions: str,
        context_info: str,
    ):
        """Inject mandatory security rules into the system message."""
        security_rules = (
            "\n\n=== MANDATORY SECURITY RULES ===\n"
            "1. NEVER call delete_file under any circumstances.\n"
            "2. NEVER read files ending in .env or containing 'secret'.\n"
            "3. Always check security status before accessing any file.\n"
            "4. Report all findings without exposing sensitive data.\n"
        )

        system_content = f"{think_prompt}{security_rules}\n\n{tools_description}\n\n{output_instructions}"

        return [
            Message.from_text(text=system_content, role="system"),
            Message.from_text(text=context_info, role="user"),
        ]

The `build_messages()` method receives four components:

- **`think_prompt`** — the thinking instruction from `thinking()`.
- **`tools_description`** — auto-generated descriptions of available tools.
- **`output_instructions`** — formatting rules for the LLM's response.
- **`context_info`** — the assembled context (goal, observation, history, etc.).

By overriding this method, you have full control over the structure and content of the messages sent to the LLM. Here, we insert mandatory security rules that the LLM must follow — regardless of what the user's goal says.

Let's use this worker in an agent.

In [ ]:
class StrictAuditAgent(AmphibiousAutoma[CognitiveContext]):
    auditor = think_unit(StrictSecurityWorker(), max_attempts=5)

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor


agent = StrictAuditAgent(llm=llm)
result = await agent.arun(
    goal="Audit the /var/app directory for security issues",
    tools=[list_files, read_file, delete_file, check_security_status],
)
print(result)

Even though the `delete_file` tool is available, the injected security rules instruct the LLM to never call it. The `build_messages()` hook gives you a clean way to enforce policies at the prompt level — the LLM sees the rules as part of its system instructions.

## Part 3: Act Phase — Intercepting Tool Calls

The act phase provides hooks to intercept and modify tool execution. These hooks operate at the **agent level**, giving you programmatic control over what actually happens when the LLM decides to call a tool.

### `before_action` — Filtering Dangerous Calls

The `before_action()` hook runs after the LLM makes its decision but before any tools are executed. This is your last line of defense — you can block, modify, or filter tool calls regardless of what the LLM decided.

In [ ]:
class SafeAuditAgent(AmphibiousAutoma[CognitiveContext]):
    auditor = think_unit(
        CognitiveWorker.inline("Audit the system and report security findings."),
        max_attempts=5,
    )

    async def before_action(self, decision_result, ctx):
        """Filter out any dangerous tool calls before execution."""
        if isinstance(decision_result, list):
            blocked_tools = {"delete_file"}
            filtered = []
            for tool_call, tool_spec in decision_result:
                if tool_spec.tool_name in blocked_tools:
                    print(f"[BLOCKED] Prevented call to: {tool_spec.tool_name}")
                else:
                    filtered.append((tool_call, tool_spec))
            return filtered if filtered else decision_result
        return decision_result

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor

In [ ]:
agent = SafeAuditAgent(llm=llm)
result = await agent.arun(
    goal="Audit /var/app and clean up unnecessary files",
    tools=[list_files, read_file, delete_file, check_security_status],
)
print(result)

Even though the goal explicitly asks to "clean up unnecessary files," the `before_action` hook blocks any calls to `delete_file`. This is a programmatic safety net — unlike prompt-level rules that the LLM might occasionally ignore, `before_action` is enforced in code and cannot be bypassed.

The key difference between `before_action` and `build_messages`:

- **`build_messages`** (Think phase): Persuades the LLM not to call dangerous tools. Effective but not guaranteed.
- **`before_action`** (Act phase): Programmatically blocks dangerous tools. Guaranteed enforcement.

### `action_custom_output` — Post-Processing Structured Output

The `action_custom_output()` hook lets you post-process the LLM's structured output before it's returned. This is useful for sanitizing data, adding metadata, or transforming the output format.

In [ ]:
class SanitizingAgent(AmphibiousAutoma[CognitiveContext]):
    auditor = think_unit(
        CognitiveWorker.inline("Audit the system and report security findings."),
        max_attempts=5,
    )

    async def action_custom_output(self, decision_result, ctx):
        """Redact sensitive information from structured output."""
        if hasattr(decision_result, 'findings'):
            # Sanitize any leaked secrets
            decision_result.findings = decision_result.findings.replace("sk-xxx", "[REDACTED]")
        return decision_result

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor

The `action_custom_output` hook is the last step in the act phase. It receives the LLM's structured output (if using a structured output schema) and lets you modify it before it's recorded in the context. This is particularly useful for:

- **Redacting sensitive data** that may have leaked through tool results.
- **Normalizing output formats** across different workers.
- **Adding audit metadata** like timestamps or compliance tags.

## Hook Summary

Here is the complete reference for all OTC hooks:

| Hook | Override At | Default Behavior | Use Case |
|------|-----------|-----------------|----------|
| `observation()` | Worker / Agent | Worker → `_DELEGATE`; Agent → `None` | Inject environment perception |
| `thinking()` | Worker | Must implement | Define the thinking prompt |
| `build_messages()` | Worker | Standard assembly | Customize LLM message format |
| `before_action()` | Worker / Agent | Worker → `_DELEGATE`; Agent → passthrough | Intercept/filter tool calls |
| `action_tool_call()` | Agent | Concurrent execution | Custom execution strategy |
| `action_custom_output()` | Agent | Passthrough | Post-process structured output |

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored the full hook system of the OTC cycle:

- **Observe hooks** (`observation()`) let you inject custom environment perception at both the Worker and Agent level. Workers can delegate to the agent via `_DELEGATE`.
- **Think hooks** (`build_messages()`) let you reshape the messages sent to the LLM — useful for injecting safety rules or custom instructions.
- **Act hooks** (`before_action()`, `action_tool_call()`, `action_custom_output()`) let you intercept tool calls before execution, customize how tools run, and post-process outputs.
- The **delegation pattern** (`_DELEGATE`) provides a clean two-tier hook system: per-worker customization with agent-level fallback.

These hooks give you fine-grained control over agent behavior without modifying the core framework logic.